# AutoBrands end-to-end matching tests

Reusable interactive checks for the production DataFrame path:

**Input DataFrame → `normalize.normalize_text` → `matching.Matcher.match_dataframe` → Results DataFrame**

The notebook loads the current catalog and production modules. It does not add matching rules or alter catalog data.

## 1. Configuration

In [1]:
INPUT_DF = None  # Existing pandas DataFrame

BRAND_COL = "brand"  # Or None
MODEL_COL = "model"  # Or None

CATALOG_PATH = "data/vehicle_catalog.csv"
TOP_K = 5
DEBUG = False


## 2. Imports and shared matcher

In [2]:
from pathlib import Path
import sys
import importlib

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "normalize.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "normalize.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

# Import the root modules as a package so their relative imports stay consistent.
normalize = importlib.import_module(f"{PROJECT_ROOT.name}.normalize")
sys.modules["normalize"] = normalize
matching = importlib.import_module(f"{PROJECT_ROOT.name}.matching") 

In [3]:
catalog_path = Path(CATALOG_PATH)
if not catalog_path.is_absolute():
    catalog_path = PROJECT_ROOT / catalog_path
if not catalog_path.is_file():
    raise FileNotFoundError(f"Vehicle catalog not found: {catalog_path}")
if isinstance(TOP_K, bool) or not isinstance(TOP_K, int) or TOP_K < 1:
    raise ValueError("TOP_K must be a positive integer")
matcher = matching.Matcher.from_csv(catalog_path, top_k=TOP_K)
catalog_df = pd.read_csv(catalog_path, dtype=str, keep_default_na=False)
catalog_pairs = {(brand.upper(), model.upper())
                 for brand, model in zip(catalog_df["brand"], catalog_df["model"])}

effective_configuration = pd.Series({
    "min_score": matcher.min_score,
    "min_margin": matcher.min_margin,
    "min_brand": matcher.min_brand,
    "min_model": matcher.min_model,
    "top_k": matcher.top_k,
}, name="effective_value")
display(effective_configuration.to_frame())
print(f"Catalog: {len(catalog_df):,} rows; schema={list(catalog_df.columns)}")


,effective_value
min_score,80.0
min_margin,8.0
min_brand,80.0
min_model,80.0
top_k,5.0


Catalog: 14,742 rows; schema=['brand', 'brand_aliases', 'model', 'model_aliases', 'vehicle_type']


## 3. Independent synthetic test datasets

The six compact datasets cover brand-only, model-only, joint, ambiguous, variant and dirty/Unicode inputs.

In [4]:
import numpy as np


brands_only = pd.DataFrame(
    {"brand": ["Toyota", "Honda", "Yamaha", "Citroen", "Volkswagen","VW", "ΜΕΡΣΕDES-BEN", "ΥΑΜΑΗΑ", "ΒΜW", "SUΖUΚΙ VΙΤΑR",
               "ΤΟΥΟΤΑ", "Unknown Maker", None, "Toyota"]},
    index=[f"brand-{i}" for i in range(14)],
)

models_only = pd.DataFrame(
    {"model": ["Yaris", "Yaris Cross", "MT-07", "PCX125", "2CV", "SUΖUΚΙ VΙΤΑR",
               "Skaravaios", "Σκαραβα$ος", "Golf +", "Unknown Model", None, "Yaris"]},
    index=[f"model-{i}" for i in range(12)],
)

brand_model = pd.DataFrame([
    {"brand": "Toyota", "model": "Yaris"},
    {"brand": "---", "model": "Yaris"},
    {"brand": "SUΖUΚΙ VΙΤΑR", "model": " "},
    {"brand": "   ", "model": "SUΖUΚΙ VΙΤΑR"},
    {"brand": np.nan,  "model": "Yaris"},
    {"brand": "Honda", "model": "PCX125"},
    {"brand": "Yamaha", "model": "MT09"},
    {"brand": "Audi", "model": "A3 Sportback"},
    {"brand": "Citroen", "model": "Vatrachos"},
    {"brand": "Toyota", "model": "Land Cruiser DIESEL"},
    {"brand": "Kawasaki", "model": "Ninja 400"},
    {"brand": "BMW", "model": "R1250GS"},
    {"brand": "Toyota", "model": "Civic"},
    {"brand": "Toyota", "model": "Yaris"},
], index=[f"pair-{i}" for i in range(14)])

ambiguities = pd.DataFrame([
    {"brand": "Volkswagen", "model": "Skaravaios"},
    {"brand": "Volkswagen", "model": "1300 Skaravaios"},
    {"brand": "Citroen", "model": "Aschimopapo"},
    {"brand": "Renault", "model": "Katrel"},
    {"brand": "Fiat", "model": "500araki"},
    {"brand": "Toyota", "model": "Civic"},
    {"brand": None, "model": "Pioneer"},
    {"brand": "Yamaha", "model": "MT-0"},
    {"brand": "Unknown Maker", "model": "Unknown Model"},
], index=[f"amb-{i}" for i in range(9)])

variants = pd.DataFrame([
    {"brand": "Yamaha", "model": "MT-07"},
    {"brand": "Yamaha", "model": "MT07"},
    {"brand": "Yamaha", "model": "MT-09SP"},
    {"brand": "Yamaha", "model": "MT-09+"},
    {"brand": "Honda", "model": "CBR600RR"},
    {"brand": "Honda", "model": "CBR 600 RR"},
    {"brand": "Toyota", "model": "C-HR"},
    {"brand": "Toyota", "model": "Yaris+"},
    {"brand": "Toyota", "model": "Yaris Cross"},
    {"brand": "Volkswagen", "model": "Golf +"},
    {"brand": "Volkswagen", "model": "Golf+"},
    {"brand": "Fiat", "model": "500-110F"},
], index=[f"variant-{i}" for i in range(12)])

dirty_data = pd.DataFrame([
    {"brand": "  toyota  ", "model": "  yaris  "},
    {"brand": "Ｔｏｙｏｔａ", "model": "Ｙａｒｉｓ"},
    {"brand": "ΤΟΥΟΤΑ", "model": "YARIS"},
    {"brand": "CITROΕN", "model": "C3"},
    {"brand": "YAMHA", "model": "MT07"},
    {"brand": "TOYTA", "model": "YARSI"},
    {"brand": None, "model": "PCX 125"},
    {"brand": "Honda", "model": None},
    {"brand": "???", "model": "@@@"},
    {"brand": "Volkswagen", "model": "Σκαραβαίος"},
    {"brand": "Citroen", "model": "Asximopapo"},
    {"brand": "  toyota  ", "model": "  yaris  "},
], index=[f"dirty-{i}" for i in range(12)])

datasets = {
    "brands_only": brands_only,
    "models_only": models_only,
    "brand_model": brand_model,
    "ambiguities": ambiguities,
    "variants": variants,
    "dirty_data": dirty_data,
}

pd.DataFrame({"rows": {name: len(frame) for name, frame in datasets.items()},
              "columns": {name: ", ".join(frame.columns) for name, frame in datasets.items()}})

,rows,columns
brands_only,14,brand
models_only,12,model
brand_model,14,"brand, model"
ambiguities,9,"brand, model"
variants,12,"brand, model"
dirty_data,12,"brand, model"


## 4. Shared normalization and matching pipeline

Only selected source columns enter the matcher. All other columns and original values stay in the returned DataFrame.

In [5]:
def run_pipeline(
    df,
    brand_col=None,
    model_col=None,
    debug=False,
):
    """Normalize selected source columns, match them, and append results."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not df.columns.is_unique:
        raise ValueError("DataFrame column names must be unique")
    if brand_col is None and model_col is None:
        raise ValueError("Supply at least one of brand_col or model_col")
    if not isinstance(debug, bool):
        raise TypeError("debug must be a bool")
    for name in (brand_col, model_col):
        if name is not None and (not isinstance(name, str) or not name):
            raise ValueError("Selected column names must be nonempty strings or None")
        if name is not None and name not in df.columns:
            raise ValueError(f"Selected input column not found: {name!r}")
    if brand_col is not None and brand_col == model_col:
        raise ValueError("brand_col and model_col must be different columns")

    result_columns = []
    if brand_col is not None:
        result_columns.append("corrected_brand")
    if model_col is not None:
        result_columns.append("corrected_model")
    result_columns.extend(["match_status", "match_score", "candidates"])
    debug_columns = ([
        "ab_brand_normalized", "ab_model_normalized", "ab_brand_score",
        "ab_model_score", "ab_score_margin", "ab_match_method",
    ] if debug else [])
    collisions = set(result_columns + debug_columns) & set(df.columns)
    if collisions:
        raise ValueError(f"Output column name collision: {', '.join(sorted(collisions))}")

    # Build only the two canonical matcher inputs; array construction avoids
    # alignment surprises when the source index contains duplicates.
    canonical = pd.DataFrame(index=df.index)
    canonical["brand"] = (df[brand_col].to_numpy(copy=False)
                          if brand_col is not None else [pd.NA] * len(df))
    canonical["model"] = (df[model_col].to_numpy(copy=False)
                          if model_col is not None else [pd.NA] * len(df))

    normalized = canonical.copy()
    for column in ("brand", "model"):
        normalized[column] = normalized[column].map(normalize.normalize_text)
    if brand_col is None:
        normalized = normalized.drop(columns="brand")
    if model_col is None:
        normalized = normalized.drop(columns="model")

    matched = matcher.match_dataframe(
        normalized,
        brand_col="brand" if brand_col is not None else None,
        model_col="model" if model_col is not None else None,
        top_k=TOP_K, debug=debug,
    )
    result = df.copy(deep=False)
    for name in result_columns + debug_columns:
        if name in matched.columns:
            result[name] = matched[name].to_numpy(copy=False)
    if len(result) != len(df) or not result.index.equals(df.index):
        raise AssertionError("Pipeline changed row count, order, or index")
    return result


## 5. Real DataFrame usage

The configured `INPUT_DF` runs when supplied. This example is guarded and runs only if `my_dataframe` exists:

```python
result = run_pipeline(
    my_dataframe,
    brand_col="manufacturer",
    model_col="vehicle_name",
)
display(result.head(20))
```

Single-column inputs are supported with `run_pipeline(df, brand_col="make")` or `run_pipeline(df, model_col="model_name")`.

In [6]:
if INPUT_DF is not None:
    configured_result = run_pipeline(
        INPUT_DF,
        brand_col=BRAND_COL,
        model_col=MODEL_COL,
        debug=DEBUG,
    )
    display(configured_result.head(20))

# The following example runs only if a DataFrame named my_dataframe exists.
if "my_dataframe" in globals() and isinstance(my_dataframe, pd.DataFrame):
    result = run_pipeline(
        my_dataframe,
        brand_col="manufacturer",
        model_col="vehicle_name",
    )
    display(result.head(20))

## 6. Results and candidate inspection

In [7]:
RESULT_COLUMNS = ["corrected_brand", "corrected_model", "match_status",
                  "match_score", "candidates"]
DEBUG_COLUMNS = ["ab_brand_normalized", "ab_model_normalized",
                 "ab_brand_score", "ab_model_score",
                 "ab_score_margin", "ab_match_method"]
FULL_MATCHES = {"EXACT_MATCH", "ALIAS_MATCH", "FUZZY_MATCH"}

def result_view(result_df: pd.DataFrame, debug: bool = False) -> pd.DataFrame:
    output_names = RESULT_COLUMNS + (DEBUG_COLUMNS if debug else [])
    source_columns = [name for name in result_df.columns if name not in output_names]
    return result_df[source_columns + [name for name in output_names
                                      if name in result_df.columns]].copy()

def show_candidates(result_df: pd.DataFrame, row_index):
    selected = result_df.loc[row_index]
    if isinstance(selected, pd.DataFrame):
        raise ValueError("row_index must identify exactly one row")
    candidates = selected["candidates"] or []
    columns = ["canonical_brand", "score"]
    if candidates and "model" in candidates[0]:
        columns.insert(1, "canonical_model")
    if not candidates:
        display(Markdown(f"No ranked candidates for row `{row_index}`."))
        return pd.DataFrame(columns=columns)
    frame = pd.DataFrame(candidates[:TOP_K]).rename(
        columns={"brand": "canonical_brand", "model": "canonical_model"}
    )[columns]
    display(frame)
    return frame

def diagnostics(result_df: pd.DataFrame) -> pd.Series:
    statuses = result_df["match_status"]
    return pd.Series({
        "total_rows": len(result_df),
        "accepted_matches": int(statuses.isin(FULL_MATCHES | {"MATCHED"}).sum()),
        "ambiguous_cases": int(statuses.eq("AMBIGUOUS").sum()),
        "unresolved_cases": int(statuses.eq("UNRESOLVED").sum()),
        "multiple_candidates": int(result_df["candidates"].map(len).gt(1).sum()),
    })

def status_counts(result_df: pd.DataFrame) -> pd.Series:
    statuses = ["MATCHED", "EXACT_MATCH", "ALIAS_MATCH", "FUZZY_MATCH",
                "PARTIAL_MATCH", "AMBIGUOUS", "CONFLICT", "UNRESOLVED"]
    return result_df["match_status"].value_counts().reindex(statuses, fill_value=0)

def accepted_matches(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].isin(FULL_MATCHES | {"MATCHED"})]

def ambiguous_results(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].eq("AMBIGUOUS")]

def unresolved_results(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].eq("UNRESOLVED")]

## 7. Execute all six synthetic datasets

In [8]:
input_snapshots = {name: frame.copy(deep=True) for name, frame in datasets.items()}
dataset_columns = {
    "brands_only": ("brand", None),
    "models_only": (None, "model"),
    "brand_model": ("brand", "model"),
    "ambiguities": ("brand", "model"),
    "variants": ("brand", "model"),
    "dirty_data": ("brand", "model"),
}
results = {
    name: run_pipeline(
        frame,
        brand_col=dataset_columns[name][0],
        model_col=dataset_columns[name][1],
    )
    for name, frame in datasets.items()
}

diagnostic_table = pd.DataFrame(
    {name: diagnostics(frame) for name, frame in results.items()}
).T
status_table = pd.DataFrame(
    {name: status_counts(frame) for name, frame in results.items()}
).T.add_prefix("status_")
display(diagnostic_table.join(status_table))
display(Markdown("All synthetic datasets were processed with the root matcher."))


,total_rows,accepted_matches,ambiguous_cases,unresolved_cases,multiple_candidates,status_MATCHED,status_EXACT_MATCH,status_ALIAS_MATCH,status_FUZZY_MATCH,status_PARTIAL_MATCH,status_AMBIGUOUS,status_CONFLICT,status_UNRESOLVED
brands_only,14,12,0,2,13,11,0,0,1,0,0,0,2
models_only,12,10,0,2,11,0,5,2,3,0,0,0,2
brand_model,14,11,2,0,14,1,6,2,2,0,2,1,0
ambiguities,9,4,1,1,9,0,0,4,0,1,1,2,1
variants,12,10,1,0,12,2,6,0,2,1,1,0,0
dirty_data,12,10,0,2,10,4,2,1,3,0,0,0,2


All synthetic datasets were processed with the root matcher.

In [9]:
for dataset_name, result_df in results.items():
    display(Markdown(f"### {dataset_name}"))
    display(result_view(result_df))

### brands_only

,brand,corrected_brand,match_status,match_score,candidates
brand-0,Toyota,TOYOTA,MATCHED,100.0,"[{'brand': 'TOYOTA', 'score': 100.0}, {'brand'..."
brand-1,Honda,HONDA,MATCHED,100.0,"[{'brand': 'HONDA', 'score': 100.0}, {'brand':..."
brand-2,Yamaha,YAMAHA,MATCHED,100.0,"[{'brand': 'YAMAHA', 'score': 100.0}, {'brand'..."
brand-3,Citroen,CITROËN,MATCHED,100.0,"[{'brand': 'CITROËN', 'score': 100.0}, {'brand..."
brand-4,Volkswagen,VOLKSWAGEN,MATCHED,100.0,"[{'brand': 'VOLKSWAGEN', 'score': 100.0}, {'br..."
brand-5,VW,VOLKSWAGEN,MATCHED,100.0,"[{'brand': 'VOLKSWAGEN', 'score': 100.0}, {'br..."
brand-6,ΜΕΡΣΕDES-BEN,MERCEDES-BENZ,MATCHED,80.2,"[{'brand': 'MERCEDES-BENZ', 'score': 80.2}, {'..."
brand-7,ΥΑΜΑΗΑ,YAMAHA,MATCHED,100.0,"[{'brand': 'YAMAHA', 'score': 100.0}, {'brand'..."
brand-8,ΒΜW,BMW,MATCHED,100.0,"[{'brand': 'BMW', 'score': 100.0}, {'brand': '..."
brand-9,SUΖUΚΙ VΙΤΑR,SUZUKI,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."


### models_only

,model,corrected_model,match_status,match_score,candidates
model-0,Yaris,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
model-1,Yaris Cross,YARIS CROSS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS CROSS', '..."
model-2,MT-07,MT-07,EXACT_MATCH,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-07', 'score'..."
model-3,PCX125,PCX125,EXACT_MATCH,100.0,"[{'brand': 'HONDA', 'model': 'PCX125', 'score'..."
model-4,2CV,AZ 2CV,ALIAS_MATCH,100.0,"[{'brand': 'CITROËN', 'model': 'AZ 2CV', 'scor..."
model-5,SUΖUΚΙ VΙΤΑR,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."
model-6,Skaravaios,BEETLE,ALIAS_MATCH,100.0,"[{'brand': 'VOLKSWAGEN', 'model': 'BEETLE', 's..."
model-7,Σκαραβα$ος,BEETLE,FUZZY_MATCH,90.0,"[{'brand': 'VOLKSWAGEN', 'model': 'BEETLE', 's..."
model-8,Golf +,GOLF,FUZZY_MATCH,80.0,"[{'brand': 'VOLKSWAGEN', 'model': 'GOLF', 'sco..."
model-9,Unknown Model,UNKNOWN MODEL,UNRESOLVED,53.8,"[{'brand': 'FORD', 'model': 'MODEL', 'score': ..."


### brand_model

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
pair-0,Toyota,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-1,---,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-2,SUΖUΚΙ VΙΤΑR,,SUZUKI,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."
pair-3,,SUΖUΚΙ VΙΤΑR,SUZUKI,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."
pair-4,NaN,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-5,Honda,PCX125,HONDA,PCX125,EXACT_MATCH,100.0,"[{'brand': 'HONDA', 'model': 'PCX125', 'score'..."
pair-6,Yamaha,MT09,YAMAHA,MT-09,MATCHED,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-09', 'score'..."
pair-7,Audi,A3 Sportback,AUDI,A3,ALIAS_MATCH,100.0,"[{'brand': 'AUDI', 'model': 'A3', 'score': 100..."
pair-8,Citroen,Vatrachos,CITROËN,DS,ALIAS_MATCH,100.0,"[{'brand': 'CITROËN', 'model': 'DS', 'score': ..."
pair-9,Toyota,Land Cruiser DIESEL,TOYOTA,LAND CRUISER DIESEL,AMBIGUOUS,79.5,"[{'brand': 'TOYOTA', 'model': 'LAND CRUISER 25..."


... (2 outputs left out)

### variants

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
variant-0,Yamaha,MT-07,YAMAHA,MT-07,EXACT_MATCH,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-07', 'score'..."
variant-1,Yamaha,MT07,YAMAHA,MT-07,MATCHED,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-07', 'score'..."
variant-2,Yamaha,MT-09SP,YAMAHA,MT-09SP,EXACT_MATCH,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-09SP', 'scor..."
variant-3,Yamaha,MT-09+,YAMAHA,MT-09+,AMBIGUOUS,89.1,"[{'brand': 'YAMAHA', 'model': 'MT-09', 'score'..."
variant-4,Honda,CBR600RR,HONDA,CBR600RR,EXACT_MATCH,100.0,"[{'brand': 'HONDA', 'model': 'CBR600RR', 'scor..."
variant-5,Honda,CBR 600 RR,HONDA,CBR600RR,MATCHED,100.0,"[{'brand': 'HONDA', 'model': 'CBR600RR', 'scor..."
variant-6,Toyota,C-HR,TOYOTA,C-HR,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'C-HR', 'score':..."
variant-7,Toyota,Yaris+,TOYOTA,YARIS+,PARTIAL_MATCH,89.1,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
variant-8,Toyota,Yaris Cross,TOYOTA,YARIS CROSS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS CROSS', '..."
variant-9,Volkswagen,Golf +,VOLKSWAGEN,GOLF,FUZZY_MATCH,87.0,"[{'brand': 'VOLKSWAGEN', 'model': 'GOLF', 'sco..."


### dirty_data

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
dirty-0,toyota,yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
dirty-1,Ｔｏｙｏｔａ,Ｙａｒｉｓ,ＴＯＹＯＴＡ,ＹＡＲＩＳ,UNRESOLVED,NaN,[]
dirty-2,ΤΟΥΟΤΑ,YARIS,TOYOTA,YARIS,MATCHED,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
dirty-3,CITROΕN,C3,CITROËN,C3,MATCHED,100.0,"[{'brand': 'CITROËN', 'model': 'C3', 'score': ..."
dirty-4,YAMHA,MT07,YAMAHA,MT-07,FUZZY_MATCH,94.2,"[{'brand': 'YAMAHA', 'model': 'MT-07', 'score'..."
dirty-5,TOYTA,YARSI,TOYOTA,YARIS,FUZZY_MATCH,84.8,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
dirty-6,NaN,PCX 125,HONDA,PCX125,MATCHED,100.0,"[{'brand': 'HONDA', 'model': 'PCX125', 'score'..."
dirty-7,Honda,NaN,HONDA,,MATCHED,100.0,"[{'brand': 'HONDA', 'score': 100.0}, {'brand':..."
dirty-8,???,@@@,,,UNRESOLVED,NaN,[]
dirty-9,Volkswagen,Σκαραβαίος,VOLKSWAGEN,BEETLE,ALIAS_MATCH,100.0,"[{'brand': 'VOLKSWAGEN', 'model': 'BEETLE', 's..."


### Inspect ranked candidates and filters

In [10]:
candidate_example = show_candidates(results["ambiguities"], "amb-0")

debug_result = run_pipeline(brand_model.head(3), brand_col="brand", model_col="model", debug=True)
display(Markdown("**Optional debug view**"))
display(result_view(debug_result, debug=True))


display(Markdown("**Ambiguous inputs**"))
display(result_view(ambiguous_results(results["ambiguities"])))
display(Markdown("**Accepted inputs**"))
display(result_view(accepted_matches(results["brand_model"])))
display(Markdown("**Unresolved inputs**"))
display(result_view(unresolved_results(results["dirty_data"])))

,canonical_brand,canonical_model,score
0,VOLKSWAGEN,BEETLE,100.0
1,VOLKSWAGEN,CARAVELLE,61.0
2,VOLKSWAGEN,CARAVELLE GL,51.2
3,VOLKSWAGEN,KUBELWAGEN,41.5
4,VOLKSWAGEN,CAMPINGWAGEN,40.4


**Optional debug view**

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates,ab_brand_normalized,ab_model_normalized,ab_brand_score,ab_model_score,ab_score_margin,ab_match_method
pair-0,Toyota,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'...",TOYOTA,YARIS,100.0,100.0,24.4,EXACT_CANONICAL
pair-1,---,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'...",---,YARIS,100.0,100.0,37.5,EXACT_CANONICAL
pair-2,SUΖUΚΙ VΙΤΑR,,SUZUKI,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score...",SUΖUΚΙ VΙΤΑR,,88.6,88.6,11.8,ORDINARY_FUZZY


**Ambiguous inputs**

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
amb-7,Yamaha,MT-0,YAMAHA,MT-0,AMBIGUOUS,87.0,"[{'brand': 'YAMAHA', 'model': 'MT-01', 'score'..."


**Accepted inputs**

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
pair-0,Toyota,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-1,---,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-2,SUΖUΚΙ VΙΤΑR,,SUZUKI,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."
pair-3,,SUΖUΚΙ VΙΤΑR,SUZUKI,VITARA,FUZZY_MATCH,88.6,"[{'brand': 'SUZUKI', 'model': 'VITARA', 'score..."
pair-4,NaN,Yaris,TOYOTA,YARIS,EXACT_MATCH,100.0,"[{'brand': 'TOYOTA', 'model': 'YARIS', 'score'..."
pair-5,Honda,PCX125,HONDA,PCX125,EXACT_MATCH,100.0,"[{'brand': 'HONDA', 'model': 'PCX125', 'score'..."
pair-6,Yamaha,MT09,YAMAHA,MT-09,MATCHED,100.0,"[{'brand': 'YAMAHA', 'model': 'MT-09', 'score'..."
pair-7,Audi,A3 Sportback,AUDI,A3,ALIAS_MATCH,100.0,"[{'brand': 'AUDI', 'model': 'A3', 'score': 100..."
pair-8,Citroen,Vatrachos,CITROËN,DS,ALIAS_MATCH,100.0,"[{'brand': 'CITROËN', 'model': 'DS', 'score': ..."
pair-11,BMW,R1250GS,BMW,R1250GS,EXACT_MATCH,100.0,"[{'brand': 'BMW', 'model': 'R1250GS', 'score':..."


**Unresolved inputs**

,brand,model,corrected_brand,corrected_model,match_status,match_score,candidates
dirty-1,Ｔｏｙｏｔａ,Ｙａｒｉｓ,ＴＯＹＯＴＡ,ＹＡＲＩＳ,UNRESOLVED,NaN,[]
dirty-8,???,@@@,,,UNRESOLVED,NaN,[]


## 8. Structural and real-DataFrame validation

In [11]:
for name, source_df in datasets.items():
    result_df = results[name]
    assert len(result_df) == len(source_df), f"{name}: row count changed"
    assert result_df.index.equals(source_df.index), f"{name}: index/order changed"
    pd.testing.assert_frame_equal(source_df, input_snapshots[name])
    brand, model = dataset_columns[name]
    expected = list(source_df.columns)
    if brand is not None:
        expected.append("corrected_brand")
    if model is not None:
        expected.append("corrected_model")
    expected += ["match_status", "match_score", "candidates"]
    assert list(result_df.columns) == expected, f"{name}: unexpected output columns"
    assert all(len(candidates) <= TOP_K for candidates in result_df["candidates"])

# Arbitrary Unicode/spaced source names, extra columns and duplicate indices.
unicode_source = pd.DataFrame({
    "Κατασκευαστής": ["ΤΟΥΟΤΑ", "Toyota", "Toyota", "???", None, "  	 "],
    "Vehicle name / model": ["Yaris", "", "Yaris", "@@@", "Σκαθάρι 1500", "  	 "],
    "claim notes": ["keep", "blank", "duplicate", "noise", "greek", "whitespace"],
}, index=[7, 7, 9, 10, 10, 11])
unicode_snapshot = unicode_source.copy(deep=True)
unicode_result = run_pipeline(
    unicode_source,
    brand_col="Κατασκευαστής",
    model_col="Vehicle name / model",
)
assert len(unicode_result) == len(unicode_source)
assert unicode_result.index.equals(unicode_source.index)
pd.testing.assert_frame_equal(unicode_source, unicode_snapshot)
assert unicode_result["claim notes"].tolist() == unicode_source["claim notes"].tolist()
assert unicode_result.loc[7, "corrected_brand"].iloc[0] == "TOYOTA"
assert unicode_result.loc[7, "corrected_brand"].iloc[1] == "TOYOTA"

# One selected field, including arbitrary names; the unselected correction is omitted.
brand_named = run_pipeline(unicode_source, brand_col="Κατασκευαστής")
model_named = run_pipeline(unicode_source, model_col="Vehicle name / model")
assert "corrected_brand" in brand_named and "corrected_model" not in brand_named
assert "corrected_model" in model_named and "corrected_brand" not in model_named

# Empty input preserves its schema and appends the selected result fields.
empty_source = pd.DataFrame(columns=["make name", "model name", "source note"])
empty_result = run_pipeline(empty_source, brand_col="make name", model_col="model name")
assert empty_result.empty and len(empty_result.columns) == 8

# Missing-column selection and result-name collisions fail before matching.
for bad_call in (
    lambda: run_pipeline(unicode_source),
    lambda: run_pipeline(unicode_source, brand_col="not here"),
    lambda: run_pipeline(pd.DataFrame({"make": ["Toyota"],
                                       "corrected_brand": ["keep"]}), brand_col="make"),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("Expected input/schema validation error")

# Configured real input is optional; synthetic datasets always use the same API.
assert INPUT_DF is None or isinstance(INPUT_DF, pd.DataFrame)
print("All DataFrame API, preservation, schema and synthetic-dataset assertions passed.")

All DataFrame API, preservation, schema and synthetic-dataset assertions passed.


## 8. Execution findings

Assertion failures indicate a pipeline or matcher defect. Unresolved deliberately unknown inputs are listed separately as potential catalog-coverage or negative-test cases.

In [12]:
coverage_rows = []
for dataset_name, result_df in results.items():
    mask = result_df["match_status"].eq("UNRESOLVED")
    for row_index, row in result_df.loc[mask].iterrows():
        coverage_rows.append({
            "dataset": dataset_name,
            "row_index": row_index,
            "brand": row.get("brand"),
            "model": row.get("model"),
        })

print("Matcher bugs detected by notebook assertions: none.")
if coverage_rows:
    display(Markdown("**Potential catalog coverage / deliberate negative cases**"))
    display(pd.DataFrame(coverage_rows))
else:
    print("Potential catalog coverage gaps among these cases: none.")

Matcher bugs detected by notebook assertions: none.


**Potential catalog coverage / deliberate negative cases**

,dataset,row_index,brand,model
0,brands_only,brand-11,Unknown Maker,NaN
1,brands_only,brand-12,NaN,NaN
2,models_only,model-9,NaN,Unknown Model
3,models_only,model-10,NaN,NaN
4,ambiguities,amb-8,Unknown Maker,Unknown Model
5,dirty_data,dirty-1,Ｔｏｙｏｔａ,Ｙａｒｉｓ
6,dirty_data,dirty-8,???,@@@
